In [297]:
import numpy as np 
import math as mt
import modern_robotics as mr

sqrt3 = float(np.sqrt(3)) 

$ PRELIMINARIES: $

In [298]:
def get_screw(w, q=None, is_prismatic=False, v_dir=None):
    w = np.array(w, dtype=float)
    if is_prismatic:
        v = np.array(v_dir, dtype=float)
    else:
        q = np.array(q, dtype=float)
        v = -np.cross(w, q)
    return np.concatenate((w, v))

sample_w = np.array([0, 0, 1])
sample_q = np.array([10, 0, 0])

result = get_screw(sample_w, sample_q)
print(result)

[  0.   0.   1.  -0. -10.  -0.]


In [299]:
M = np.array([
    [ 1, 0, 0, 2+sqrt3],
    [ 0, 1, 0, 0],
    [ 0, 0, 1, 1+sqrt3],
    [ 0, 0, 0, 1] 
], dtype=float)

print("----- END-EFFECTOR ZERO POSITION (M) -----")
print(M)

----- END-EFFECTOR ZERO POSITION (M) -----
[[1.     0.     0.     3.7321]
 [0.     1.     0.     0.    ]
 [0.     0.     1.     2.7321]
 [0.     0.     0.     1.    ]]


$ SCREW-AXIS CODE: $

In [300]:
# Helper function
def get_screw(w, q=None, is_prismatic=False, v_dir=None):
    w = np.array(w, dtype=float)
    if is_prismatic:
        v = np.array(v_dir, dtype=float)
    else:
        q = np.array(q, dtype=float)
        v = -np.cross(w, q)
    return np.concatenate((w, v))

# 3. Space Screw Axes in {0} (S_i)
w1 = [0, 0, 1];   q1 = [1, 0, 0]
w2 = [0, 1, 0];   q2 = [1, 0, 0]
w3 = [0, 1, 0];   q3 = [1 + sqrt3, 0, -1]
w4 = [0, 1, 0];   q4 = [2 + sqrt3, 0, sqrt3 - 1]
w5 = [0, 0, 0];   v_dir5 = [0, 0, 1] # Prismatic
w6 = [0, 0, 1];   q6 = [2 + sqrt3, 0, 1 + sqrt3]

S1 = get_screw(w1, q=q1)
S2 = get_screw(w2, q=q2)
S3 = get_screw(w3, q=q3)
S4 = get_screw(w4, q=q4)
S5 = get_screw(w5, is_prismatic=True, v_dir=v_dir5)
S6 = get_screw(w6, q=q6)

# Set print formatting to make the arrays easy to read (max 4 decimal places)
np.set_printoptions(precision=4, suppress=True)

print("--- Space Screw Axes (S_i) in {0} ---")
for i, S in enumerate([S1, S2, S3, S4, S5, S6], start=1):
    print(f"{S}")
print("\n")

# 4. Body Screw Axes in {b} (B_i)
# Fixed typo here: replaced " + sqrt(3)" with "sqrt3"
p_b = np.array([2 + sqrt3, 0, 1+sqrt3], dtype=float)

qb1 = np.array(q1, dtype=float) - p_b
qb2 = np.array(q2, dtype=float) - p_b
qb3 = np.array(q3, dtype=float) - p_b
qb4 = np.array(q4, dtype=float) - p_b
qb6 = np.array(q6, dtype=float) - p_b

B1 = get_screw(w1, q=qb1)
B2 = get_screw(w2, q=qb2)
B3 = get_screw(w3, q=qb3)
B4 = get_screw(w4, q=qb4)
B5 = get_screw(w5, is_prismatic=True, v_dir=v_dir5)
B6 = get_screw(w6, q=qb6)

print("--- Body Screw Axes (B_i) in {b} ---")
for i, B in enumerate([B1, B2, B3, B4, B5, B6], start=1):
    print(f"{B}")

--- Space Screw Axes (S_i) in {0} ---
[ 0.  0.  1. -0. -1. -0.]
[ 0.  1.  0. -0. -0.  1.]
[ 0.      1.      0.      1.     -0.      2.7321]
[ 0.      1.      0.     -0.7321 -0.      3.7321]
[0. 0. 0. 0. 0. 1.]
[ 0.      0.      1.     -0.     -3.7321 -0.    ]


--- Body Screw Axes (B_i) in {b} ---
[ 0.      0.      1.      0.      2.7321 -0.    ]
[ 0.      1.      0.      2.7321 -0.     -2.7321]
[ 0.      1.      0.      3.7321 -0.     -1.    ]
[ 0.  1.  0.  2. -0. -0.]
[0. 0. 0. 0. 0. 1.]
[ 0.  0.  1. -0. -0. -0.]


In [301]:
S_list = np.column_stack([S1, S2, S3, S4, S5, S6])

# 3. Joint Variables (theta) - Ensure dtype=float
theta_list = np.array([-np.pi/2, np.pi/2, np.pi/3, -np.pi/4, 1.0, np.pi/6], dtype=float)

# 4. Compute Forward Kinematics in Space Frame
T = mr.FKinSpace(M, S_list, theta_list)

# Output the result rounded to 4 decimal places (exceeds the 0.01 error requirement)
np.set_printoptions(precision=4, suppress=True)
print("End-Effector Configuration T in SE(3):")
print(T)

End-Effector Configuration T in SE(3):
[[ 0.5     0.866   0.      1.    ]
 [ 0.2241 -0.1294 -0.9659 -1.8978]
 [-0.8365  0.483  -0.2588 -4.5085]
 [ 0.      0.      0.      1.    ]]


In [302]:
B_list = np.column_stack([B1, B2, B3, B4, B5, B6])

# 3. Joint Variables (theta) - Ensure dtype=float
theta_list = np.array([-np.pi/2, np.pi/2, np.pi/3, -np.pi/4, 1.0, np.pi/6], dtype=float)

# 4. Compute Forward Kinematics in Space Frame
T1 = mr.FKinBody(M, B_list, theta_list)

# Output the result rounded to 4 decimal places (exceeds the 0.01 error requirement)
np.set_printoptions(precision=4, suppress=True)
print("End-Effector Body T in SE(3):")
print(T1)

End-Effector Body T in SE(3):
[[ 0.5     0.866   0.      1.    ]
 [ 0.2241 -0.1294 -0.9659 -1.8978]
 [-0.8365  0.483  -0.2588 -4.5085]
 [ 0.      0.      0.      1.    ]]


[[1,0,0,3.73], [0,1,0,0], [0,0,1,2.73], [0,0,0,1]]

[[0,0,1,0,-1,0],[0,1,0,0,0,1],[0,1,0,1,0,2.73],[0,1,0,-0.73,0,3.73],[0,0,0,0,0,1],[0,0,1,0,-3.73,0]]

[[0,0,1,0,2.73,0],[0,1,0,2.73,0,-2.73],[0,1,0,3.73,0,-1],[0,1,0,2,0,0],[0,0,0,0,0,1],[0,0,1,0,0,0]]

[[0.5,0.866,0,1],[0.2241,-0.1294,-0.9659,-1.8978],[-0.8365,0.483,-0.2588,-4.5085],[0,0,0,1]]

In [1]:
import numpy as np

def calculate_joint_torques():
    # 1. Define the spatial wrench (moment_z, force_x, force_y)
    # The tip generates 2 N in x_s, 0 in y_s, and 0 moment
    F_s = np.array([0, 2, 0])
    
    # 2. Define the Spatial Jacobian (J_s)
    # Columns are the screw axes S_1, S_2, S_3
    sqrt2_over_2 = np.sqrt(2) / 2
    
    J_s = np.array([
        [1, 1, 1],
        [0, 0, sqrt2_over_2],
        [0, -1, -1 - sqrt2_over_2]
    ])
    
    # 3. Calculate torques using tau = J_s^T * F_s
    tau = J_s.T @ F_s
    
    # Round to 3 decimal places for clean output
    tau_rounded = np.round(tau, 3)
    
    return tau_rounded

# Execute and print
torques = calculate_joint_torques()
print(f"The required joint torques (tau_1, tau_2, tau_3) are: {tuple(torques)}")

The required joint torques (tau_1, tau_2, tau_3) are: (np.float64(0.0), np.float64(0.0), np.float64(1.414))


In [304]:


# 1. Define Home Configuration M (4x4 SE(3) representation for the library)
# Tip at (4,0,0) at home
M = np.array([[1, 0, 0, 4],
            [0, 1, 0, 0],
            [0, 0, 1, 0],
            [0, 0, 0, 1]])

# 2. Define Space Screw Axes S (planar joints along x-axis)
Slist = np.array([
    [0, 0, 1, 0,  0, 0], # Joint 1 at (0,0)
    [0, 0, 1, 0, -1, 0], # Joint 2 at (1,0)
    [0, 0, 1, 0, -2, 0], # Joint 3 at (2,0)
    [0, 0, 1, 0, -3, 0]  # Joint 4 at (3,0)
]).T

# 3. Current Configuration (Part C)
thetalist = np.array([0, 0, np.pi/2, -np.pi/2])

# 4. Body Wrench Fb
Fb = np.array([0, 0, 10, 10, 10, 0])

# 5. Step-by-Step Calculation
# Transform Space Screws to Body Screws
Blist = mr.Adjoint(np.linalg.inv(M)) @ Slist

# Calculate Body Jacobian at current theta
Jb = mr.JacobianBody(Blist, thetalist)

# Calculate Torques: tau = Jb^T * Fb
tau = Jb.T @ Fb

print(f"Torques (tau1, tau2, tau3, tau4): {np.round(tau, 4)}")

Torques (tau1, tau2, tau3, tau4): [30. 20. 10. 20.]
